In [ ]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

import ast

from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder


from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import shap
from sklearn.feature_extraction.text import TfidfVectorizer

from sentence_transformers import SentenceTransformer

import seaborn as sns


In [ ]:
df_movies = pd.read_parquet("../data/movies_filtered_cleaned.parquet")
df_ratings = pd.read_parquet("../data/processed_df_ratings_filtered.parquet")
df_users = pd.read_parquet("../data/all_users_stats_post_movies_filter.parquet")

In [ ]:
# df_movies.sample(3)

In [ ]:
# df_movies.columns

In [ ]:
# df_ratings.sample(4)

In [ ]:
# df_ratings.columns

In [ ]:
# df_users.sample(2)

In [ ]:
# df_users.columns

In [ ]:
# tfidf = TfidfVectorizer(max_features=100)  # or 300, depending on data size
# tfidf_matrix = tfidf.fit_transform(df_movies['overview'].fillna(''))

# tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=[f'overview_tfidf_{i}' for i in range(tfidf_matrix.shape[1])])
# df_movies = pd.concat([df_movies.reset_index(drop=True), tfidf_df], axis=1)

# tfidf_cols = [col for col in df_movies.columns if col.startswith('overview_tfidf_')]

# features_for_cluster = tfidf_cols + [
#     'vote_average', 'popularity_score', 'runtime', 'release_decade', 'genre_count'
# ]

# df_movies['genre_list'] = df_movies['genre_list'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# # mlb = MultiLabelBinarizer()
# # genre_multi_hot = mlb.fit_transform(df_movies['genre_list'])
# # genre_cols = [f"genre_{g}" for g in mlb.classes_]
# # genre_df = pd.DataFrame(genre_multi_hot, columns=genre_cols)

# # df_movies = pd.concat([df_movies.reset_index(drop=True), genre_df.reset_index(drop=True)], axis=1)

# # features_for_cluster += genre_cols

# df_movies = pd.get_dummies(df_movies, columns=['main_genre'])
# main_genre_cols = [col for col in df_movies.columns if col.startswith('main_genre_')]
# features_for_cluster += main_genre_cols

# X = df_movies[features_for_cluster].fillna(0)
# X_scaled = StandardScaler().fit_transform(X)


In [ ]:
df_movies.sample(2)

In [ ]:
# [x for x in df_movies.columns if "tfidf" not in x]

In [ ]:

# kmeans = KMeans(n_clusters=20, random_state=0)
# df_movies['cluster'] = kmeans.fit_predict(X_scaled)

In [ ]:
# df_movies['cluster']

In [ ]:
# import matplotlib.pyplot as plt
# from sklearn.cluster import KMeans
# from sklearn.preprocessing import StandardScaler

# # Assuming X is your feature matrix (e.g., including TF-IDF and genres), and is already cleaned
# X_scaled = StandardScaler().fit_transform(X)

# inertia = []
# # cluster_range = range(2, 251)  # Try between 2 and 30 clusters
# cluster_range = range(2, 151)  # Try between 2 and 30 clusters

# for k in cluster_range:
#     kmeans = KMeans(n_clusters=k, random_state=0, n_init=10)
#     kmeans.fit(X_scaled)
#     inertia.append(kmeans.inertia_)

# plt.figure(figsize=(14,10))
# plt.plot(cluster_range, inertia, marker='o')
# plt.xlabel('Number of clusters (k)')
# plt.ylabel('Inertia (within-cluster sum of squares)')
# plt.title('Elbow Method for Optimal k')
# plt.grid()
# plt.show()


In [ ]:
# from sklearn.decomposition import NMF

# nmf = NMF(n_components=20, random_state=0)
# overview_topics = nmf.fit_transform(tfidf_matrix)  # tfidf_matrix = TF-IDF for 'overview'
# topic_cols = [f"topic_{i}" for i in range(20)]
# overview_topics_df = pd.DataFrame(overview_topics, columns=topic_cols)
# df_movies = pd.concat([df_movies.reset_index(drop=True), overview_topics_df.reset_index(drop=True)], axis=1)

# # 2. Combine with genres, numerics, etc.
# features_for_cluster = topic_cols + main_genre_cols + [
#     'vote_average', 'vote_count', 'popularity_score', 'critical_success', 'runtime'
# ]

# # 3. Optionally, reduce with PCA
# from sklearn.preprocessing import StandardScaler
# from sklearn.decomposition import PCA
# X = df_movies[features_for_cluster].fillna(0)
# X_scaled = StandardScaler().fit_transform(X)
# pca = PCA(n_components='mle')
# X_pca = pca.fit_transform(X_scaled)

# # # 4. KMeans
# from sklearn.cluster import KMeans
# kmeans = KMeans(n_clusters=60, random_state=0)
# df_movies['cluster'] = kmeans.fit_predict(X_pca)

In [ ]:
# len(df_movies['cluster'].unique())

In [ ]:
# [x for x in df_movies.columns if "topic" not in x]

In [ ]:


# 1. TF-IDF vectorization (if not already done)
tfidf = TfidfVectorizer(max_features=100, stop_words='english')
tfidf_matrix = tfidf.fit_transform(df_movies['overview'].fillna(''))

# 2. NMF topic modeling
n_topics = 45
nmf = NMF(n_components=n_topics, random_state=0)
overview_topics = nmf.fit_transform(tfidf_matrix)
topic_cols = [f"topic_{i}" for i in range(n_topics)]
overview_topics_df = pd.DataFrame(overview_topics, columns=topic_cols)

# Add to df_movies
df_movies = pd.concat([df_movies.reset_index(drop=True), overview_topics_df.reset_index(drop=True)], axis=1)

# Optional: Print top words for each topic
feature_names = tfidf.get_feature_names_out()
for idx, topic in enumerate(nmf.components_):
    top_words = [feature_names[i] for i in topic.argsort()[-8:][::-1]]
    print(f"Topic {idx}: {', '.join(top_words)}")


# One-hot encode main_genre
df_movies = pd.get_dummies(df_movies, columns=['main_genre'])

# Collect the new main_genre columns
main_genre_cols = [col for col in df_movies.columns if col.startswith('main_genre_')]

# Example: Add to your feature list for clustering or modeling
features_for_cluster = (
    topic_cols + main_genre_cols +
    [
		'release_year',
		'release_month',
		'release_decade',
		'movie_age',
		'runtime',
		'popularity', 'vote_average',
		'vote_min',
		'vote_max',
		'vote_count',
		'popularity_score',
		'critical_success',
		'crowd_approval',
		'lead_actor_popularity',
		'director_popularity',
    ]
)

X = df_movies[features_for_cluster].fillna(0)
X_scaled = StandardScaler().fit_transform(X)
pca = PCA(n_components=20, random_state=0)
X_pca = pca.fit_transform(X_scaled)


from sklearn.cluster import KMeans

n_clusters = 60  # Pick based on elbow/silhouette, or start with 25
kmeans = KMeans(n_clusters=n_clusters, random_state=0)
df_movies['cluster'] = kmeans.fit_predict(X_pca)


In [ ]:
# Print summary for each cluster
for cluster_id in range(n_clusters):
    print(f"\n=== Cluster {cluster_id} ===")
    cluster_movies = df_movies[df_movies['cluster'] == cluster_id]

    # Top genres in the cluster
    top_genres = cluster_movies[main_genre_cols].sum().sort_values(ascending=False).head(5)
    print("Top genres:", ', '.join([g.replace('genre_', '') for g in top_genres.index]))

    # Average values for numerical features
    print("Avg vote average:", cluster_movies['vote_average'].mean())
    print("Avg popularity score:", cluster_movies['popularity_score'].mean())
    print("Release decades:", cluster_movies['release_decade'].value_counts().head(3).to_dict())

    # Top topics (overview) in the cluster
    topic_means = cluster_movies[topic_cols].mean().sort_values(ascending=False)
    for i in topic_means.head(3).index:
        topic_idx = int(i.split('_')[-1])
        top_words = [feature_names[j] for j in nmf.components_[topic_idx].argsort()[-8:][::-1]]
        print(f"Top topic {topic_idx}: {', '.join(top_words)}")


# Recommender system


In [ ]:
len(df_ratings["userId"].unique())

In [ ]:
df_ratings = df_ratings.merge(df_movies[['movieId', 'cluster']], on='movieId', how='left')

user_cluster_scores = (
    df_ratings.groupby(['userId', 'cluster'])['rating']
    .mean()
    .reset_index()
    .rename(columns={'rating': 'cluster_mean_rating'})
)

user_id = 4
top_clusters = user_cluster_scores[user_cluster_scores['userId'] == user_id] \
    .sort_values('cluster_mean_rating', ascending=False)['cluster'].head(3).tolist()

seen_movies = df_ratings[df_ratings['userId'] == user_id]['movieId']
recommend_pool = df_movies[~df_movies['movieId'].isin(seen_movies) & df_movies['cluster'].isin(top_clusters)]

recommend_pool = recommend_pool.sort_values('popularity_score', ascending=False).head(10)  # Top 10 recommendations


In [ ]:
recommend_pool

In [ ]:
user_cluster_scores


In [ ]:
movie_id = 1221  # example
movie_row = df_movies[df_movies['movieId'] == movie_id]
cluster_id = int(movie_row['cluster'].iloc[0])


# Example: summarize cluster with top genres and topics
cluster_movies = df_movies[df_movies['cluster'] == cluster_id]
# Find the main_genre one-hot columns
main_genre_cols = [col for col in cluster_movies.columns if col.startswith('main_genre_')]
# Sum up each genre's count in this cluster
top_genres_count = cluster_movies[main_genre_cols].sum().sort_values(ascending=False).head(3)
# To print just genre names (remove 'main_genre_' prefix):
top_genres_names = [col.replace('main_genre_', '') for col in top_genres_count.index]


top_topics = []
for i in topic_cols:
    mean_val = cluster_movies[i].mean()
    top_topics.append((i, mean_val))
top_topics = sorted(top_topics, key=lambda x: x[1], reverse=True)[:3]
# For topic words, you already have code to print them for each cluster


user_id = 123  # example
user_ratings = df_ratings[df_ratings['userId'] == user_id].copy()

user_cluster_means = user_ratings.groupby('cluster')['rating'].mean()
user_top_clusters = user_cluster_means.sort_values(ascending=False).head(3).index.tolist()
user_affinity = user_cluster_means.get(cluster_id, None)

print(f"We recommend '{movie_row['title'].iloc[0]}' because you tend to rate movies highly from cluster {cluster_id}.")
print("This cluster is mostly:")
print(f"- Genres: {', '.join([g for g in top_genres_count.index])}")
print(f"- Typical release decades: {', '.join(map(str, cluster_movies['release_decade'].value_counts().head(2).index))}")
print("Thematic topics include:")

top_words_for_topic = {}

for idx, topic in enumerate(nmf.components_):
    top_words = [feature_names[i] for i in topic.argsort()[-8:][::-1]]
    top_words_for_topic[idx] = top_words  # Save for later!
    print(f"Topic {idx}: {', '.join(top_words)}")


if user_affinity:
    print(f"You give movies from this cluster an average rating of {user_affinity:.2f}.")

for topic_idx, _ in top_topics:
    topic_number = int(topic_idx.split('_')[1])
    print(f"  - Topic {topic_number}: {', '.join(top_words_for_topic[topic_number])}")



In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.bar(recommend_pool['title'], recommend_pool['popularity_score'])
plt.xticks(rotation=75, ha='right')
plt.title('Top Recommended Movies by Popularity Score')
plt.ylabel('Popularity Score')
plt.xlabel('Movie Title')
plt.tight_layout()
plt.show()


In [ ]:

main_genre_cols = [col for col in recommend_pool.columns if col.startswith('main_genre_')]
recommend_genres = recommend_pool[main_genre_cols].sum().sort_values(ascending=False)

nonzero_recommend_genres = recommend_genres[recommend_genres > 0]

plt.figure(figsize=(8,4))
sns.barplot(
    x=nonzero_recommend_genres.values,
    y=[g.replace('main_genre_', '') for g in nonzero_recommend_genres.index]
)
plt.title('Genre Distribution in Recommendations')
plt.xlabel('Count')
plt.ylabel('Genre')
plt.show()


In [ ]:
import plotly.graph_objects as go

# Get all main genre columns (in case some are zero in recommend_pool)
main_genre_cols = [col for col in recommend_pool.columns if col.startswith('main_genre_')]
all_genre_names = [g.replace('main_genre_', '') for g in main_genre_cols]

# For genres not present, ensure a count of 0
all_counts = recommend_pool[main_genre_cols].sum().reindex(main_genre_cols, fill_value=0).values

fig = go.Figure(
    go.Barpolar(
        r=all_counts,
        theta=all_genre_names,
        marker_line_color="black",
        marker_line_width=2,
        opacity=0.8
    )
)
fig.update_layout(
    title="Genre Distribution in Recommendations (Radial Chart, All Genres)",
    polar=dict(
        radialaxis=dict(showticklabels=True, ticks=''),
    ),
    showlegend=False
)
fig.show()


In [ ]:
cluster_counts = recommend_pool['cluster'].value_counts().sort_index()

plt.figure(figsize=(6,6))
plt.pie(cluster_counts, labels=cluster_counts.index, autopct='%1.1f%%')
plt.title('Cluster Distribution in Recommendations')
plt.show()


In [ ]:
# Plot topic weights for one recommended movie vs. cluster average
movie = recommend_pool.iloc[0]
cluster_id = movie['cluster']
cluster_avg = df_movies[df_movies['cluster'] == cluster_id][topic_cols].mean()

plt.figure(figsize=(10,3))
plt.plot(range(len(topic_cols)), [movie[col] for col in topic_cols], label='This Movie')
plt.plot(range(len(topic_cols)), cluster_avg.values, label='Cluster Avg')
plt.title('Topic Distribution: This Movie vs. Cluster Avg')
plt.xlabel('Topic')
plt.ylabel('Weight')
plt.legend()
plt.show()


In [ ]:
# inertia = []
# # cluster_range = range(2, 251)  # Try between 2 and 30 clusters
# cluster_range = range(2, 151)  # Try between 2 and 30 clusters

# for k in cluster_range:
#     kmeans = KMeans(n_clusters=k, random_state=0, n_init=10)
#     kmeans.fit(X_scaled)
#     inertia.append(kmeans.inertia_)

# plt.figure(figsize=(14,10))
# plt.plot(cluster_range, inertia, marker='o')
# plt.xlabel('Number of clusters (k)')
# plt.ylabel('Inertia (within-cluster sum of squares)')
# plt.title('Elbow Method for Optimal k')
# plt.grid()
# plt.show()
